# ForgeLM — GaLore Memory Optimization

Train with **GaLore** (Gradient Low-Rank Projection) — an optimizer-level
technique that projects gradients into a low-rank subspace so the
optimizer state (Adam's moment buffers) shrinks by the same factor.
Combines cleanly with LoRA: ForgeLM applies GaLore to the LoRA adapter's
trainable parameters, giving you LoRA's parameter-efficiency *and*
GaLore's optimizer-state savings in one pass.

**Implementation note (different from the academic GaLore paper):**

- The academic paper applies GaLore to **all** model weights for
  full-parameter training and produces no adapter file.
- ForgeLM requires a `lora:` block in every config (Phase 19 library-API
  invariant), so this notebook trains a LoRA adapter and uses GaLore as
  its **optimizer** — gradient projection is applied to the adapter
  parameters, not the frozen base weights. Output is a PEFT adapter
  loadable via `PeftModel.from_pretrained` (see Step 7).
- For pure full-parameter GaLore, use `optim: "galore_adamw_8bit"`
  directly on a non-LoRA HF trainer — that path is outside ForgeLM's
  scope.

**What you save with `galore_adamw_8bit` (this notebook):** 8-bit
optimizer states cut Adam's first+second moment buffers from FP32 (8
bytes/param) to int8 (1 byte/param) — ~75% reduction on the LoRA
adapter's optimizer state. Useful on tight-VRAM Colab T4.

**Requirements:** GPU runtime required (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/galore_memory_optimization.ipynb)

In [ ]:
# Step 1: Install ForgeLM + GaLore
# Pinned to v0.5.7; bump on each release
!pip install -q --no-cache-dir 'forgelm[qlora]==0.5.7' bitsandbytes
!pip install -q galore-torch  # GaLore optimizer
!pip uninstall -y wandb -q
# Colab base image ships torchao 0.10; modern peft requires >=0.16 for PeftModel.from_pretrained.
# Upgrade so the comparison cell at the end can load the trained LoRA adapter.
!pip install -q --upgrade 'torchao>=0.16.0'
!forgelm --version

In [ ]:
# Step 2: Check GPU and verify GaLore is installed
import importlib.util

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"GPU: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No GPU. GaLore requires GPU.")
    print("Go to Runtime → Change runtime type → T4 GPU")

if importlib.util.find_spec("galore_torch"):
    print("GaLore: installed")
else:
    print("ERROR: galore-torch not installed. Run: pip install galore-torch")

In [ ]:
# Step 3: Create GaLore config
# Note: GaLore is configured in the training section, not the lora section
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 512
  load_in_4bit: false  # GaLore doesn't need 4-bit quantization

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  output_dir: "./galore_checkpoints"
  max_steps: 50
  per_device_train_batch_size: 2
  learning_rate: 1.0e-5

  # GaLore configuration
  galore_enabled: true
  galore_optim: "galore_adamw_8bit"  # 8-bit variant saves even more memory
  galore_rank: 64                      # lower rank = less memory, may reduce quality
  galore_update_proj_gap: 200          # steps between SVD re-computations
  galore_scale: 0.25                   # gradient scaling factor

data:
  dataset_name_or_path: "timdettmers/openassistant-guanaco"
  shuffle: true
"""

with open("galore_config.yaml", "w") as f:
    f.write(config_yaml)
print("GaLore config saved!")
print("  Optimizer: galore_adamw_8bit")
print("  Rank: 64 (gradient projection dimension)")
print("  Scale: 0.25")

In [ ]:
# Step 4: Validate config
!forgelm --config galore_config.yaml --dry-run

In [ ]:
# Step 5: Train with GaLore
!forgelm --config galore_config.yaml

In [ ]:
# Step 6: Verify output
import os

model_path = "./galore_checkpoints/final_model"
if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("ERROR: Model not found. Check training output above.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")

In [ ]:
# Step 7: Compare base vs GaLore-trained model
import os

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./galore_checkpoints/final_model"
base_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("Error: Model not found. Ensure training completed.")
else:
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    print("Loading GaLore-trained model...")
    ft_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base_model_name), model_path)
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)

    test_prompts = [
        "What is the difference between AI and machine learning?",
        "How do I get started with data science?",
        "Explain cloud computing in simple terms.",
    ]

    for prompt in test_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt")

        print(f"\n{'=' * 60}")
        print(f"PROMPT: {prompt}")
        print(f"{'=' * 60}")

        base_out = base_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        base_resp = tokenizer.decode(base_out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[BASE MODEL]:\n{base_resp[:400]}")

        ft_inputs = ft_tokenizer(formatted, return_tensors="pt")
        ft_out = ft_model.generate(**ft_inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        ft_resp = ft_tokenizer.decode(ft_out[0][ft_inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[GALORE-TRAINED]:\n{ft_resp[:400]}")

    print(f"\n{'=' * 60}")
    print("GaLore updates all weights (full-parameter) while using less optimizer memory.")
    print("For 50 steps the difference may be subtle — increase max_steps for stronger effect.")

## GaLore vs LoRA Comparison (as configured in ForgeLM)

Both are enabled simultaneously in this notebook's config — LoRA defines
*which parameters train*, GaLore defines *how their gradients update*.

| Aspect | LoRA alone | LoRA + GaLore (this notebook) |
|--------|------------|-------------------------------|
| What trains | LoRA adapter matrices (~0.2% of params) | Same — LoRA adapter matrices |
| Optimizer state | FP32 Adam (8 bytes/param) | 8-bit Adam projected to low-rank (~1 byte/param × low-rank) |
| Output file | PEFT adapter | PEFT adapter (identical format — load via `PeftModel.from_pretrained`) |
| Inference overhead | Adapter loading/merging | Same as LoRA |
| When to add GaLore | When optimizer-state VRAM dominates your peak | Long sequences, big batch, or large LoRA `r` on tight-VRAM cards |

**For pure full-parameter GaLore** (no LoRA adapter, what the paper
describes): see the upstream `galore-torch` README. ForgeLM's API is
optimised for parameter-efficient fine-tuning so it always wraps with
LoRA.

## GaLore Optimizer Variants

| Variant | Use Case |
|---------|----------|
| `galore_adamw` | Standard — good balance |
| `galore_adamw_8bit` | Less memory — recommended for Colab |
| `galore_adafactor` | Experimental — alternative optimizer |
| `galore_adamw_layerwise` | Lowest memory — single GPU only, no DDP |